# Introduction to RAG

Large language models (LLMs) can write fluently about almost any topic — but fluency is not the same as **knowing your data**. A model trained on public web text does not automatically know your internal API documentation, yesterday's incident postmortem, or the exact refund policy stored in a PDF on SharePoint.

When users ask questions about **private**, **fresh**, or **domain-specific** knowledge, a bare LLM must rely on **parametric memory** (patterns frozen in weights at training time). That often produces confident answers that are **outdated**, **invented**, or **unverifiable**.

**Retrieval-Augmented Generation (RAG)** addresses this by connecting the LLM to an external **knowledge base** at inference time. The system **retrieves** relevant passages, **augments** the prompt with that evidence, and **generates** an answer conditioned on retrieved context — turning the model from a general narrator into an **evidence synthesizer**.

The idea was popularized in research (Lewis et al., 2020, *Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks*) and is now the default architecture for document-grounded assistants, support bots, and enterprise search-with-answers.

This notebook defines RAG, explains parametric vs non-parametric memory, contrasts RAG with fine-tuning and long-context prompting, maps the offline/online pipeline, previews failure modes, and runs demonstrations that show why retrieval matters before generation.

Prerequisites: **02. GenAI & LLM Fundamentals** (especially hallucinations and tokens), **03. LLM API Integration**, **04. Prompt Engineering**, and **05. Embeddings & Vector DBs** (chunking, similarity search, Chroma).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
np.random.seed(42)
plt.style.use("seaborn-v0_8-whitegrid")

---

## 1. The Knowledge Gap Problem

An LLM at inference time does two things: read your prompt and **predict likely next tokens**. It does **not** query a database of facts unless you build that connection.

### 1.1 Parametric Memory

**Parametric memory** is everything the model "knows" because it was encoded in billions of weights during pre-training and alignment. It is powerful for general reasoning, language, and common knowledge — but bounded:

| Limitation | What goes wrong | Example |
|------------|-----------------|--------|
| **Training cutoff** | New facts after training are absent | "What changed in API v3.2 released last month?" |
| **No private corpora** | Internal docs never seen | HR leave policy, proprietary schemas |
| **No guaranteed recall** | Likely text ≠ verified fact | Invented CLI flag that sounds plausible |
| **No citations** | User cannot audit the answer | "According to our docs…" with no source |
| **Confident hallucination** | Smooth prose masks errors | Wrong Alembic command that almost works |

The model's training objective is **next-token likelihood**, not **truth verification**. A prompt that *looks like* a question the model has seen may get a plausible continuation — whether or not that continuation matches your organization's reality.

### 1.2 What Users Actually Need

Document-grounded applications usually require:

- **Freshness** — answers reflect the latest published docs
- **Grounding** — claims trace to specific passages
- **Control** — you decide what the system is allowed to cite
- **Updateability** — change docs without retraining a model

RAG supplies **non-parametric memory**: facts stored outside the weights, fetched per query.

---

## 2. What Is RAG?

**Retrieval-Augmented Generation** is an **architecture pattern**, not a single Python package. It combines:

1. A **retriever** — finds relevant text from a knowledge base
2. An **augmenter** — inserts retrieved text into the LLM prompt
3. A **generator** — the LLM produces an answer using that context

At a high level:

$$\text{answer} = \text{LLM}\big(\text{prompt}(q,\; \text{Retrieve}(q,\; \mathcal{D}))\big)$$

where $q$ is the user query and $\mathcal{D}$ is your indexed document collection.

### 2.1 The Name: Retrieve → Augment → Generate

| Step | Verb | What happens |
|------|------|--------------|
| **Retrieve** | Search | Map $q$ to top-k chunks from $\mathcal{D}$ |
| **Augment** | Stuff / compose | Add chunks + instructions to the prompt |
| **Generate** | Complete | LLM writes the final answer |

RAG does **not** replace the LLM. The LLM still does language understanding and synthesis. Retrieval **narrows** what the model should talk about.

### 2.2 RAG Is Not "Search Only"

Classic search returns links or snippets. RAG **synthesizes** — it may combine three chunks into one coherent paragraph, rephrase for the user's level, or refuse when evidence is insufficient (if prompted correctly). The generation step is what makes it conversational.

---

## 3. Parametric vs Non-Parametric Memory

Thinking in two memory types clarifies design trade-offs:

```
┌─────────────────────────────────────────────────────────────┐
│                    LLM INFERENCE                            │
│  ┌─────────────────────┐    ┌─────────────────────────────┐ │
│  │ Parametric memory   │    │ Non-parametric memory (RAG) │ │
│  │ (model weights)     │    │ (vector DB, files, APIs)    │ │
│  │                     │    │                             │ │
│  │ Grammar, reasoning, │    │ Your PDFs, tickets, wikis,  │ │
│  │ broad world knowledge│   │ runbooks, code comments     │ │
│  └─────────────────────┘    └─────────────────────────────┘ │
│            ▲                              ▲                 │
│            │                              │                 │
│       fixed at train time          updated per ingest      │
└─────────────────────────────────────────────────────────────┘
```

| Memory type | Update cost | Best for |
|-------------|-------------|----------|
| **Parametric** | Retrain / fine-tune | Style, format, reasoning habits |
| **Non-parametric** | Re-index documents | Facts, policies, API fields |

Strong assistants use **both**: the LLM provides language and reasoning; the knowledge base provides **evidence**.

In [ ]:
# Illustrative: parametric vs non-parametric update latency (toy units)
strategies = ["Edit doc + re-index", "Swap prompt template", "Fine-tune model"]
hours_to_deploy = np.array([0.5, 1.0, 48.0])  # illustrative, not measured

plt.figure(figsize=(7, 3.5))
bars = plt.barh(strategies, hours_to_deploy, color=["#2ca02c", "#1f77b4", "#d62728"])
plt.xlabel("Illustrative time to reflect a factual change (hours, log scale)")
plt.xscale("log")
plt.title("Updating facts: non-parametric vs parametric")
for b, v in zip(bars, hours_to_deploy):
    plt.text(v * 1.1, b.get_y() + b.get_height() / 2, f"{v}h", va="center")
plt.tight_layout()
plt.show()

Re-indexing a changed document is typically **hours or minutes**. Changing a model's factual habits via fine-tuning is **days to weeks** and still does not guarantee recall of every detail.

---

## 4. RAG vs Fine-Tuning vs Long Context vs Prompt-Only

Teams often ask: "Should we RAG, fine-tune, or just use a bigger context window?" The answer is often **a combination**, but each lever solves different problems.

### 4.1 Comparison Table

| Approach | What you change | Factual freshness | Citations | Cost driver |
|----------|-----------------|-------------------|-----------|-------------|
| **Prompt-only** | Instructions in the prompt | Low (model priors only) | None | Prompt + completion tokens |
| **RAG** | Retrieved docs in prompt | High (if index is current) | Possible | Retrieval + large context |
| **Long context** | Paste entire corpus in prompt | Medium (if it fits) | Manual | Very large input tokens |
| **Fine-tuning** | Model weights | Low unless retrained | None | Training + inference |

### 4.2 When Each Approach Wins

| Scenario | Recommended starting point |
|----------|---------------------------|
| 500-page handbook, changes weekly | **RAG** |
| Consistent JSON extraction format | **Prompt engineering**, then maybe fine-tune |
| 10-page policy, rare updates | **Long context** may suffice |
| Brand voice at millions of requests | **Fine-tune** + RAG for facts |
| Live stock price right now | **Tool/API call** (agents — notebook **12**) |

### 4.3 Stacking Patterns

Production stacks are common:

```
System prompt (tone, safety)
    + Retrieved chunks (facts)
    + Chat history (multi-turn — notebook 11)
    + Tool results (live data — notebook 12)
    → Chat completion
```

Prompt engineering from **04. Prompt Engineering** is still essential — RAG only changes **what text** sits in the context, not **how** you instruct the model to use it.

In [ ]:
approaches = ["Prompt-only", "RAG", "Long context", "Fine-tune"]
freshness = np.array([0.15, 0.90, 0.75, 0.25])
citations = np.array([0.05, 0.85, 0.40, 0.10])
ops_burden = np.array([0.20, 0.55, 0.45, 0.90])

x = np.arange(len(approaches))
w = 0.25
plt.figure(figsize=(9, 4))
plt.bar(x - w, freshness, width=w, label="Data freshness")
plt.bar(x, citations, width=w, label="Citation support")
plt.bar(x + w, ops_burden, width=w, label="Ops complexity")
plt.xticks(x, approaches)
plt.title("Illustrative trade-offs (not benchmark measurements)")
plt.ylim(0, 1.05)
plt.legend(loc="upper left")
plt.tight_layout()
plt.show()

---

## 5. The RAG Pipeline: Offline and Online

RAG systems split into **indexing** (usually batch/offline) and **querying** (real-time/online).

### 5.1 Offline — Build the Knowledge Base

```
Raw sources (PDF, MD, HTML, tickets, code)
        │
        ▼
   Parse & clean
        │
        ▼
   Chunk documents          ← 05.04 Document Chunking
        │
        ▼
   Embed each chunk         ← 05.03 Embedding APIs
        │
        ▼
   Store vectors + metadata ← 05.05–07 Vector DBs
```

Indexing runs when documents change — on deploy, nightly, or on webhook. It is **not** repeated per user question.

### 5.2 Online — Answer a Question

```
User query q
        │
        ▼
   Embed q (same model as index)
        │
        ▼
   Vector search → top-k chunks
        │
        ▼
   Assemble prompt (system + context + q)  ← 06.05 Prompting
        │
        ▼
   LLM chat completion                     ← 03. LLM API Integration
        │
        ▼
   Answer (+ optional citations)
```

### 5.3 Latency Budget (Conceptual)

$$T_{\text{total}} \approx T_{\text{embed}} + T_{\text{search}} + T_{\text{LLM}}$$

| Stage | Typical note |
|-------|--------------|
| $T_{\text{embed}}$ | One small API call for the query |
| $T_{\text{search}}$ | Milliseconds–tens of ms with ANN indexes |
| $T_{\text{LLM}}$ | Often dominates for long answers or big context |

Notebook **10** covers caching embeddings, async pipelines, and streaming.

---

## 6. The Four Engineering Stages in Detail

### 6.1 Index

Turn source files into searchable chunks with stable ids. Decisions include chunk size, overlap, metadata (source path, section title, ACL), and embedding model version. **Version** the index when any of these change.

### 6.2 Retrieve

Given $q$, return $\{d_1, \ldots, d_k\}$ — the $k$ most relevant chunks. Default: cosine similarity in embedding space (**05.02**). Extensions: metadata filters (**05.07**), hybrid BM25 + vector (**07**), reranking (**06**).

### 6.3 Augment

Format retrieved text for the LLM — delimiters, source labels, ordering, and strict instructions ("answer only from context; say I don't know if missing"). Poor augmentation wastes good retrieval.

### 6.4 Generate

Call the chat API with `temperature` tuned for factuality (often 0 for support bots). The model **reads** context at inference; weights are not updated.

| Stage | Primary failure if misconfigured |
|-------|----------------------------------|
| Index | Answer text split across chunks or never embedded |
| Retrieve | Right doc exists but not in top-k |
| Augment | Model ignores or misreads stuffed context |
| Generate | Model elaborates beyond evidence |

---

## 7. Where RAG Sits in This Curriculum

Section **05. Embeddings & Vector DBs** teaches everything **up to** the LLM box. Section **06** (this folder) teaches **wiring retrieval into generation**.

```
05 Embeddings & Vector DBs          06 RAG (this section)
────────────────────────────        ────────────────────────────
Embeddings, cosine, chunking   →    Naive pipeline (02)
Chroma / Pinecone / pgvector   →    Prompting + context (05–06)
Recall@k retrieval eval        →    End-to-end RAG eval (09)
Production index patterns      →    Production RAG (10)
```

| Prior section | What RAG reuses |
|---------------|----------------|
| **03. LLM API Integration** | Chat completions, streaming, API keys |
| **04. Prompt Engineering** | System prompts, format constraints |
| **05. Embeddings & Vector DBs** | Chunk, embed, store, search |

You should not need to re-learn cosine similarity here — you **apply** it inside a full assistant.

---

## 8. When to Use RAG

### 8.1 Strong Fit

| Use case | Why RAG works |
|----------|---------------|
| **Internal support bot** | Answers from runbooks and API docs |
| **Legal / compliance Q&A** | Ground in policy PDFs with citations |
| **Engineering onboarding** | Explain codebase from indexed modules |
| **Sales enablement** | Product sheets update without retraining |
| **Research assistant** | Synthesize across papers or notes |

### 8.2 Weak Fit

| Situation | Better alternative |
|-----------|-------------------|
| Answer requires **live transactional state** | Database query or tool (notebook **12**) |
| Entire corpus fits in context **and** rarely changes | Long-context prompt |
| Task is pure **style/format** with no external facts | Prompt engineering or fine-tune |
| User needs **creative writing** without sources | Bare LLM |
| Corpus is tiny (5 FAQ entries) | Few-shot examples in prompt may suffice |

RAG shines when knowledge is **too large to paste**, **changes over time**, or must be **attributed** to sources.

---

## 9. System Components and Decisions

Building RAG is assembling components — each with trade-offs covered in later notebooks:

| Component | Key decisions | Notebook |
|-----------|---------------|----------|
| **Corpus** | Sources, PII, access control | **04** |
| **Chunking** | Size, overlap, structure-aware splits | **05.04** |
| **Embeddings** | Model, dimensions, batching | **05.03** |
| **Vector store** | Chroma, pgvector, Pinecone | **05.06–08** |
| **Retriever** | $k$, filters, hybrid, rerank | **06**, **07** |
| **Prompt** | Instructions, delimiters, citations | **05** (this section) |
| **LLM** | Model, temperature, max tokens | **03**, **02.09** |
| **Eval** | Retrieval + generation metrics | **05.09**, **09** |

Treat these as **versioned artifacts**: `index_v3 + embed_model_small + prompt_v2 + gpt-4o-mini`.

---

## 10. Naive RAG Baseline

The **naive RAG** pipeline is the minimum viable assistant — everything else extends it:

1. **Ingest once** — embed all chunks; store in Chroma (or similar)
2. **On query** — embed the question; `query(n_results=k)`
3. **Prompt** — system message + concatenated chunks + user question
4. **Generate** — single chat completion; return text

```python
# Pseudocode — full code in notebook 02
chunks = retrieve(query, k=3)
context = "\n\n".join(chunks)
messages = [
    {"role": "system", "content": "Answer using only the context below."},
    {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {query}"},
]
answer = client.chat.completions.create(model=CHAT_MODEL, messages=messages)
```

Naive RAG assumes: same embedding model for index and query, fixed $k$, no reranking, no query rewriting, no conversation memory. That is enough to learn the pattern — and enough to fail on hard queries until you add techniques from notebooks **06–08**.

---

## 11. Failure Modes: Retrieval vs Generation

Bad RAG answers have **two independent failure layers**. Fixing the wrong layer wastes time.

```
Bad answer
    │
    ├─ Was the right chunk in top-k? ──No──► Fix chunking, embeddings, k, hybrid (05, 07)
    │
    └─ Yes ──► Did the LLM use it correctly? ──No──► Fix prompt, temperature, citations (05, 08)
```

| Symptom | Likely layer | Where to learn |
|---------|--------------|----------------|
| Wrong topic, plausible prose | Retrieval miss | **05**, **07** |
| Right chunks, wrong conclusion | Generation / prompt | **05**, **08** |
| Invented citation | Prompt / post-check | **05**, **08** |
| "I don't know" too often | $k$ too low or strict prompt | **06**, **08** |
| Answer from stale doc | Index not refreshed | **04**, **10** |

**05.09 Evaluating Retrieval Quality** measures the left branch. **09. Evaluating RAG End-to-End** measures the full pipeline.

---

## 12. Cost and Token Budget

RAG increases **input tokens** because retrieved chunks occupy the context window.

$$\text{cost per query} \approx c_{\text{in}} \cdot (T_{\text{prompt}} + T_{\text{context}} + T_{\text{question}}) + c_{\text{out}} \cdot T_{\text{answer}}$$

| Knob | Effect |
|------|--------|
| Smaller $k$ | Less context, cheaper, may miss evidence |
| Smaller chunks | More precise retrieval, more boundary cuts |
| Compression / summarization | Fewer tokens, risk of losing detail (**06**) |
| Caching frequent queries | Lower embed + LLM cost (**10**) |

A support bot retrieving five 400-token chunks adds **~2,000 input tokens** per question before the user even finishes typing their follow-up. Budget accordingly.

In [ ]:
k_values = np.array([1, 3, 5, 8])
tokens_per_chunk = 400
prompt_overhead = 250
context_tokens = k_values * tokens_per_chunk + prompt_overhead

plt.figure(figsize=(7, 4))
plt.plot(k_values, context_tokens, marker="o", linewidth=2)
plt.xlabel("top-k chunks retrieved")
plt.ylabel("Approx. input tokens (context + overhead)")
plt.title("Context size grows linearly with k")
plt.grid(True, alpha=0.3)
for k, t in zip(k_values, context_tokens):
    plt.annotate(f"{t}", (k, t), textcoords="offset points", xytext=(0, 8), ha="center")
plt.tight_layout()
plt.show()

---

## 13. Mental Model for Debugging

When a RAG answer is wrong, walk this checklist **in order**:

1. **Source** — Is the fact in the corpus at all?
2. **Chunking** — Is it split across chunks or buried in noise?
3. **Retrieval** — Is the right chunk in top-k? (log ids + scores)
4. **Filters** — Did metadata filters exclude the doc?
5. **Prompt** — Does the model see context clearly? Any conflicting instructions?
6. **Generation** — Is temperature too high for factual mode?
7. **Eval** — Is this a one-off or a pattern on a labeled set?

Logging **retrieved chunk ids** alongside each answer makes postmortems fast. If logs show the correct chunk was retrieved but the answer is wrong, stop tuning embeddings.

---

## 14. Sample Corpus for This Section

Notebooks in **06. RAG** reuse the same small knowledge base as **05. Embeddings** hands-on apps — a fictional **FastAPI Notes API** project:

| id | Content summary | `source` metadata |
|----|-----------------|-------------------|
| c1 | FastAPI in-memory Notes API | api-docs |
| c2 | Alembic `upgrade head` | db-docs |
| c3 | JWT Bearer authentication | security-docs |
| c4 | Pytest fixtures / conftest.py | test-docs |
| c5 | Alembic autogenerate | db-docs |

Using one corpus across sections lets you compare **retrieval-only** metrics (**05.09**) with **full RAG answers** (**09**).

---

## 15. Demonstration Setup

Cells below use a **toy lexical retriever** (no API) to show retrieval behavior, then optional OpenAI calls for embedding-based retrieval. Replace the placeholder API key before running live demos.

In [ ]:
OPENAI_API_KEY = "sk-proj-placeholder-replace-with-your-real-key"

from openai import OpenAI

client = OpenAI(api_key=OPENAI_API_KEY)
EMBED_MODEL = "text-embedding-3-small"

CORPUS = [
    {"id": "c1", "text": "The Notes API is built with FastAPI and stores notes in memory for demos."},
    {"id": "c2", "text": "Alembic applies SQLAlchemy schema migrations. Run alembic upgrade head after pulling new revisions."},
    {"id": "c3", "text": "JWT bearer tokens authenticate API requests. Clients send Authorization: Bearer token header."},
    {"id": "c4", "text": "Pytest fixtures share database setup across tests. Use conftest.py for session-scoped engines."},
    {"id": "c5", "text": "Alembic autogenerate compares SQLAlchemy models to the live database schema and drafts revision files."},
]


def lexical_score(query: str, doc: str) -> int:
    """Toy overlap count — fails on paraphrase (why we use embeddings)."""
    q = set(query.lower().split())
    d = set(doc.lower().split())
    return len(q & d)


def retrieve_lexical(query: str, k: int = 2) -> list[dict]:
    scored = sorted(CORPUS, key=lambda c: lexical_score(query, c["text"]), reverse=True)
    return scored[:k]


print("Ready.")

---

## 16. Demonstration: Lexical Retrieval Misses Paraphrases

Keyword overlap is a useful baseline — and a lesson in why RAG systems use embeddings.

In [ ]:
queries = [
    "How do I run database migrations?",
    "forgot my login credentials",
    "ECONNREFUSED pytest",
]

for q in queries:
    print(f"\nQuery: {q}")
    hits = retrieve_lexical(q, k=2)
    for h in hits:
        print(f"  [{h["id"]}] score={lexical_score(q, h["text"])}  {h["text"][:70]}...")

"How do I run database migrations?" may rank Alembic chunks poorly with pure overlap if the query words do not appear literally. **Embeddings** (section **05**) fix paraphrase — RAG assumes you already have a working retriever.

---

## 17. Demonstration: Stuffing Context into a Prompt

Augmentation is string assembly — order and delimiters matter for model attention.

In [ ]:
query = "How do I apply schema migrations after pulling new code?"
hits = retrieve_lexical(query, k=2)

context_blocks = []
for h in hits:
    context_blocks.append(f"[source: {h['id']}]\n{h['text']}")

context = "\n\n---\n\n".join(context_blocks)

user_message = (
    "Use ONLY the context below to answer. If the answer is not in the context, say you do not know.\n\n"
    f"### Context\n{context}\n\n### Question\n{query}"
)

print("--- Augmented user message (preview) ---")
print(user_message[:900])
print("...") if len(user_message) > 900 else None

Notebook **05. Prompting and Context Injection** formalizes system prompts, citation formats, and anti-hallucination instructions.

---

## 18. Demonstration: Embedding-Based Retrieval Preview

Same corpus, semantic similarity — the retriever used in production RAG.

In [ ]:
def embed_texts(texts: list[str]) -> list[list[float]]:
    r = client.embeddings.create(model=EMBED_MODEL, input=texts)
    ordered = sorted(r.data, key=lambda x: x.index)
    return [d.embedding for d in ordered]


def cosine(a, b) -> float:
    va, vb = np.array(a), np.array(b)
    return float(np.dot(va, vb) / (np.linalg.norm(va) * np.linalg.norm(vb)))


query = "How do I run database migrations?"
doc_texts = [c["text"] for c in CORPUS]
q_vec, *doc_vecs = embed_texts([query] + doc_texts)

scores = [(CORPUS[i]["id"], cosine(q_vec, doc_vecs[i])) for i in range(len(CORPUS))]
scores.sort(key=lambda x: x[1], reverse=True)

print(f"Query: {query}\n")
print("Top chunks by embedding similarity:")
for cid, sc in scores[:3]:
    text = next(c["text"] for c in CORPUS if c["id"] == cid)
    print(f"  {cid}  {sc:.4f}  {text[:65]}...")

Expect **c2** and **c5** (Alembic) near the top for migration questions — even without the word "migration" in every chunk. That retrieved context is what the LLM reads in notebook **02**.

---

## 19. Demonstration: Bare LLM vs Grounded Answer (Conceptual)

Without retrieval, the model guesses from priors. With retrieval, it synthesizes from provided evidence. The cell below **simulates** the difference without calling chat — useful when offline.

In [ ]:
def simulate_bare_llm(query: str) -> str:
    """Illustrative: generic prior — not a real API call."""
    priors = {
        "migration": "Run your framework's migration command (e.g. Django migrate or Flyway).",
        "auth": "Most APIs use API keys or OAuth2.",
    }
    if "migration" in query.lower():
        return "[ungounded] " + priors["migration"]
    return "[ungounded] Generic best-practice answer — may not match your stack."


def simulate_grounded_rag(query: str, hits: list[dict]) -> str:
    evidence = " ".join(h["text"] for h in hits)
    return f"[grounded] Based on retrieved docs: {evidence[:200]}..."


q = "How do I apply schema migrations after pulling new code?"
hits = retrieve_lexical(q, k=2)
print("Without RAG:", simulate_bare_llm(q))
print("\nWith RAG (evidence injected):", simulate_grounded_rag(q, hits))

The grounded path names **Alembic** and **upgrade head** when those chunks are retrieved. The ungrounded path may mention the wrong ecosystem entirely.

---

## 20. Architecture Patterns Preview

This section progresses from naive RAG to richer patterns (notebook **03**):

| Pattern | Extra behavior |
|---------|----------------|
| **Naive RAG** | Single retrieve → generate |
| **Conversational RAG** | Rewrite query with chat history (**11**) |
| **Corrective RAG** | Grade retrieval; re-query if poor (**03**, **08**) |
| **Agentic RAG** | LLM decides when/what to retrieve (**12**) |

Start simple. Measure. Add complexity only when eval shows a clear bottleneck.

---

## 21. Common Mistakes

| Mistake | Consequence | Fix |
|---------|-------------|-----|
| Skipping retrieval eval | Tune prompts while chunks are wrong | **05.09**, **09** |
| Different embed model for index vs query | Broken similarity | Same model + dimensions |
| Huge chunks | Vague retrieval, wasted tokens | Smaller structured chunks |
| No "I don't know" instruction | Hallucination on empty context | **05**, **08** |
| Stale index | Correct-looking wrong answers | Versioned re-ingest (**04**, **10**) |
| Ignoring ACL metadata | Leak restricted docs into answers | Filter at query time (**05.07**) |
| Treating RAG as search | Users expect synthesis + prose | Design for generation quality too |

---

## 22. Summary

RAG connects LLMs to **non-parametric memory** — your documents, indexed and retrieved per query. The pattern is **retrieve → augment → generate**: embed and search for relevant chunks, insert them into a carefully designed prompt, and let the model synthesize an answer grounded in that evidence.

It complements parametric memory (model weights) rather than replacing it. Choose RAG when knowledge is **private**, **changing**, or must be **cited**; combine with prompt engineering, long context, fine-tuning, or tools when those fit better.

Section **05** taught how to build the index and measure retrieval. This section teaches how to **use** retrieval in a complete assistant — starting with the naive pipeline in notebook **02**.

Demonstrations showed lexical retrieval limits, prompt stuffing, embedding-based ranking, token budget vs $k$, and the debugging split between retrieval and generation failures.

Next: **02. Naive RAG Pipeline** — implement ingest, retrieve, prompt, and chat completion end-to-end with Chroma and OpenAI.